# QC HMM Challenger

Diagnostic-only market regime challenger. This notebook must not feed A3 parameter selection, A3 freeze, runtime activation, or A4 policy choice. It answers whether market states give independent evidence around macro-release transitions.

In [ ]:
import numpy as np
import pandas as pd
from hmmlearn import hmm


Use QuantBook/History only for the challenger market features. Do not use QC FRED for the core A3 parity path.

In [ ]:
# In QC Research, uncomment and run:
# qb = QuantBook()
# symbols = [qb.add_equity(ticker).symbol for ticker in ["SPY", "IEF", "TIP", "LQD"]]
# history = qb.history(symbols, start=pd.Timestamp("2014-02-19"), end=pd.Timestamp("2026-06-24"), resolution=Resolution.DAILY)
# close = history["close"].unstack(level=0).dropna(how="all")

# Local/offline placeholder: load an exported market primitive panel if desired.
close = None


In [ ]:
def build_hmm_features(close: pd.DataFrame) -> pd.DataFrame:
    returns = close.pct_change()
    spy = returns.iloc[:, 0]
    features = pd.DataFrame(index=returns.index)
    features["spy_return"] = spy
    features["spy_vol_21d"] = spy.rolling(21).std()
    if returns.shape[1] > 1:
        features["ief_return"] = returns.iloc[:, 1]
    if returns.shape[1] > 2:
        features["tip_ief_relative"] = returns.iloc[:, 2] - returns.iloc[:, 1]
    if returns.shape[1] > 3:
        features["lqd_spy_relative"] = returns.iloc[:, 3] - spy
    return features.dropna()

def fit_hmm_challenger(features: pd.DataFrame, n_components: int = 3):
    model = hmm.GaussianHMM(n_components=n_components, covariance_type="full", n_iter=200, random_state=7)
    model.fit(features.to_numpy())
    states = model.predict(features.to_numpy())
    return model, pd.Series(states, index=features.index, name=f"hmm_{n_components}_state")


In [ ]:
if close is not None:
    features = build_hmm_features(close)
    model, states = fit_hmm_challenger(features, n_components=3)
    display(states.value_counts().sort_index())
else:
    print("Load QC History or an exported market panel to run the diagnostic HMM challenger.")
